In [41]:
import os, re, json
import PyPDF2, pytesseract, exifread
from PIL import Image
import cv2
import numpy as np
from imagededup.methods import PHash
from pdf2image import convert_from_path
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


In [43]:
def read_pdf(file_path):
    text = ""
    metadata = {}
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
        if reader.metadata:
            metadata = {k.strip("/"): str(v) for k, v in dict(reader.metadata).items()}
    return text, metadata

def read_image(file_path):
    img = Image.open(file_path)
    text = pytesseract.image_to_string(img)
    metadata = {"format": img.format, "mode": img.mode, "size": img.size}
    return text, metadata

def load_certificate(file_path):
    ext = os.path.splitext(file_path)[-1].lower()
    if ext == ".pdf":
        return read_pdf(file_path)
    elif ext in [".png", ".jpg", ".jpeg"]:
        return read_image(file_path)
    else:
        raise ValueError("Unsupported file type")


In [45]:
def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    thresh = cv2.dilate(thresh, kernel, iterations=1)
    return thresh

def detect_event_from_text(ocr_text):
    likely_events = []
    event_words = ["celebration", "hackathon", "competition", "workshop", "conference", "summit", "innovation", "fest", "expo", "baja", "tech", "fair", "meet"]
    
    for line in ocr_text.splitlines():
        line_clean = line.strip()
        if not line_clean:
            continue
        if any(word in line_clean.lower() for word in event_words):
            likely_events.append(line_clean)
        elif line_clean.isupper() and len(line_clean) > 3:
            likely_events.append(line_clean)
        elif any(char.isdigit() for char in line_clean) and any(char.isalpha() for char in line_clean):
            likely_events.append(line_clean)
        elif len(line_clean.split()) > 2:
            likely_events.append(line_clean)
    
    return list(set(likely_events))  


def analyze_id_card_pdf(pdf_path):
    try:
        pages = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        return {"ocr_snippet": "", "detected_events": [], "event_match": False}

    all_detected_events = []
    ocr_snippet_full = ""
    
    for page in pages:
        img = cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR)
        preprocessed = preprocess_image(img)
        ocr_text = pytesseract.image_to_string(preprocessed, config='--oem 3 --psm 6')
        
        ocr_snippet_full += ocr_text + "\n"
        detected_events = detect_event_from_text(ocr_text)
        all_detected_events.extend(detected_events)
        
    all_detected_events = list(set(all_detected_events))
    
    result = {
        "ocr_snippet": ocr_snippet_full[:500],
        "detected_events": all_detected_events,
        "event_match": len(all_detected_events) > 0
    }
    
    return result


In [47]:
def score_certificate(cert_text, metadata, user_claim):
    text = cert_text.lower()
    scores = {}
    scores["event"] = 30 if user_claim["event"].lower() in text else 10
    scores["institution"] = 30 if user_claim["institution"].lower() in text else 10
    scores["prize"] = 25 if user_claim["prize"].lower() in text else 5
    if user_claim["cash_prize"] and str(user_claim["cash_prize"]).lower() in text:
        scores["cash_prize"] = 15
    else:
        scores["cash_prize"] = 0

    if metadata:
        scores["metadata"] = 10
    else:
        scores["metadata"] = 0

    return scores


In [49]:
def preprocess_for_ocr(image_path):
    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    scale_percent = 150
    width = int(thresh.shape[1] * scale_percent / 100)
    height = int(thresh.shape[0] * scale_percent / 100)
    resized = cv2.resize(thresh, (width, height))

    return resized


In [51]:
def analyze_photo(file_path, event_keywords):
    try:
        processed = preprocess_for_ocr(file_path)
        ocr_text = pytesseract.image_to_string(processed, config="--psm 6").lower()
        event_match = any(keyword.lower() in ocr_text for keyword in event_keywords)

        try:
            img = Image.open(file_path)
            exif = img.getexif()
            date = exif.get(36867, "unknown")
            gps = exif.get(34853, {})
        except:
            date, gps = "unknown", {}

        return {
            "ocr_snippet": ocr_text[:100],  
            "date": date,
            "gps": gps,
            "event_match": event_match
        }
    except Exception as e:
        return {"error": str(e)}


In [53]:
def detect_duplicates_from_files(file_list):
    phasher = PHash()
    encodings = {}
    for f in file_list:
        encodings[f] = phasher.encode_image(image_file=f)
    duplicates = phasher.find_duplicates(encoding_map=encodings, max_distance_threshold=5)
    return duplicates


In [55]:
def compute_final_score(cert_scores, photo_results, id_result=None, debug=False):
    breakdown = {}

    cert_score = sum(cert_scores.get(k, 0) for k in ["event", "institution", "prize", "cash_prize", "metadata"])
    cert_score_capped = min(cert_score, 50)
    breakdown["certificate"] = cert_score_capped

    photo_score = 0
    matches = 0
    for p in photo_results:
        if p.get("event_match"):
            photo_score += 10
            matches += 1
        if p.get("gps"):
            photo_score += 5
        if p.get("date") and p["date"] != "unknown":
            photo_score += 3

    if matches >= 2:
        photo_score += 5

    photo_score_capped = min(photo_score, 30)
    breakdown["photo"] = photo_score_capped

    id_score_capped = 0
    if id_result:
        id_score = 0
        if id_result.get("event_match"):
            id_score += 10
        if id_result.get("institution_match"):
            id_score += 5
        if id_result.get("student_match"):
            id_score += 5

        id_score_capped = min(id_score, 20)
    breakdown["id_card"] = id_score_capped

    final_score = min(cert_score_capped + photo_score_capped + id_score_capped, 100)

    if debug:
        return final_score, breakdown
    return final_score


In [57]:
cert_file = "Certificates.pdf"
cert_text, cert_meta = load_certificate(cert_file)

user_input = {
    "event": "E-Baja'25",
    "institution": "SAEINDIA",
    "prize": "Participant",
    "cash_prize": "-"
}

cert_scores = score_certificate(cert_text, cert_meta, user_input)

photo_files = ["b1.jpg", "b2.jpg"]
photo_results = [analyze_photo(f, user_input["event"]) for f in photo_files]
duplicates = detect_duplicates_from_files(photo_files)

id_card_file = "cid.pdf"  
id_result = analyze_id_card_pdf(id_card_file)

final_score, breakdown = compute_final_score(cert_scores, photo_results, id_result, debug=True)

f_result = {
    "final_confidence_score": final_score,
    "score_breakdown": breakdown,
    "certificate_scores": cert_scores,
    "photo_results": photo_results,
    "id_card_result": id_result,
    "duplicate_photos": duplicates
}

print(json.dumps(f_result, indent=2))

2025-09-20 17:50:50,807: INFO Start: Evaluating hamming distances for getting duplicates
2025-09-20 17:50:50,808: INFO Start: Retrieving duplicates using BKTree algorithm
100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.01s/it]
2025-09-20 17:50:57,700: INFO End: Retrieving duplicates using BKTree algorithm
2025-09-20 17:50:57,700: INFO End: Evaluating hamming distances for getting duplicates


{
  "final_confidence_score": 60,
  "score_breakdown": {
    "certificate": 35,
    "photo": 25,
    "id_card": 0
  },
  "certificate_scores": {
    "event": 10,
    "institution": 10,
    "prize": 5,
    "cash_prize": 0,
    "metadata": 10
  },
  "photo_results": [
    {
      "ocr_snippet": "~ 4 if\nheatk see we my = wi a y\n\\ whadty wicd \u201cbb: moe ota a\nrae meo ts br ade\u2019 soiled a/////\nae a e",
      "date": "unknown",
      "gps": {},
      "event_match": true
    },
    {
      "ocr_snippet": "a. es aa ad here pe 5 : py p. e, $ tittitth\na a wee ye et bale a ee \u201cle mat |\nag fiia | be hesaeas a",
      "date": "unknown",
      "gps": {},
      "event_match": true
    }
  ],
  "id_card_result": {
    "ocr_snippet": "",
    "detected_events": [],
    "event_match": false
  },
  "duplicate_photos": {
    "b1.jpg": [],
    "b2.jpg": []
  }
}
